#Actividad 5: Entrenamiento y evaluación de modelos

Comparación de regresión logística y Random Forest para predecir el abandono de clientes.

## Importación de bibliotecas

In [1]:
from pathlib import Path
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import cross_validate, GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Carga del dataset

In [2]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/"
    "elehache11/Actividad5/main/"
    "datos/datos_limp/telco_churn_limpio.csv"
)

df = pd.read_csv(url)

print("Dataset cargado correctamente.")
print("Registros:", df.shape[0])
print("Columnas:", df.shape[1])

Dataset cargado correctamente.
Registros: 7043
Columnas: 20


## Separación de datos

In [3]:
#Separar variables predictoras y variable objetivo
X = df.drop(columns = ["Churn"])
y = df["Churn"]

#Dividir los datos: 80% entrenamiento y 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify = y
)

#Mostrar resultados
print("DIMENSIONES DE LOS CONJUNTOS")
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

print("\nDISTRIBUCIÓN DE CHURN")
print("Dataset original:")
print(y.value_counts(normalize = True).round(4))

print("\nEntrenamiento:")
print(y_train.value_counts(normalize = True).round(4))

print("\nPrueba:")
print(y_test.value_counts(normalize = True).round(4))

DIMENSIONES DE LOS CONJUNTOS
Entrenamiento: (5634, 19)
Prueba: (1409, 19)

DISTRIBUCIÓN DE CHURN
Dataset original:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64

Entrenamiento:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64

Prueba:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64


## Preprocesamiento

In [5]:
#Identificar las variables numéricas y categóricas
numericas = X.select_dtypes(include = ["number"]).columns.tolist()
categoricas = X.select_dtypes(include = ["object"]).columns.tolist()

print("\nVARIABLES NUMÉRICAS")
print(numericas)

print("\nVARIABLES CATEGÓRICAS")
print(categoricas)

#Definir las transformaciones
preprocesamiento = ColumnTransformer(
    transformers = [
        ("num", StandardScaler(), numericas),
        ("cat", OneHotEncoder(handle_unknown = "ignore"), categoricas)
    ]
)

print("\nPreprocesamiento configurado correctamente.")


VARIABLES NUMÉRICAS
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

VARIABLES CATEGÓRICAS
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Preprocesamiento configurado correctamente.


## Entrenamiento inicial

In [6]:
#Crear modelo de regresión logística
modelo_logistico = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", LogisticRegression(max_iter = 1000, random_state = 42))
])

#Crear modelo Random Forest
modelo_forest = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", RandomForestClassifier(n_estimators = 100, random_state = 42))
])

#Entrenar los modelos
print("\nENTRENAMIENTO DE MODELOS")

modelo_logistico.fit(X_train, y_train)
print("Regresión logística: entrenamiento completado.")

modelo_forest.fit(X_train, y_train)
print("Random Forest: entrenamiento completado.")


ENTRENAMIENTO DE MODELOS
Regresión logística: entrenamiento completado.
Random Forest: entrenamiento completado.


## Evaluación inicial

In [7]:
#Evaluar los modelos
print("\nEVALUACIÓN DE MODELOS")

modelos = {
    "Regresión logística": modelo_logistico,
    "Random Forest": modelo_forest
}

resultados = []

for nombre, modelo in modelos.items():

    #Generar predicciones
    predicciones = modelo.predict(X_test)

    #Calcular probabilidades de abandono
    probabilidades = modelo.predict_proba(X_test)[:, 1]

    #Calcular métricas
    accuracy = accuracy_score(y_test, predicciones)
    precision = precision_score(y_test, predicciones)
    recall = recall_score(y_test, predicciones)
    f1 = f1_score(y_test, predicciones)
    roc_auc = roc_auc_score(y_test, probabilidades)

    #Guardar resultados
    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc
    })

#Crear tabla comparativa
tabla = pd.DataFrame(resultados)

print("\nCOMPARACIÓN DE RESULTADOS")
print(tabla.round(4).to_string(index = False))


EVALUACIÓN DE MODELOS

COMPARACIÓN DE RESULTADOS
             Modelo  Accuracy  Precision  Recall     F1  ROC-AUC
Regresión logística    0.8055     0.6572  0.5588 0.6040   0.8421
      Random Forest    0.7864     0.6254  0.4866 0.5474   0.8185


## Validación cruzada

In [8]:
#Configurar validación cruzada de 5 particiones
kfold = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

#Definir las métricas
metricas = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

#Ejecutar validación cruzada
print("\nVALIDACIÓN CRUZADA (5-FOLD)")

resultados_cv = []

for nombre, modelo in modelos.items():

    evaluacion = cross_validate(
        clone(modelo),
        X_train,
        y_train,
        cv = kfold,
        scoring=metricas
    )

    #Calcular promedio y desviación estándar
    fila = {"Modelo": nombre}

    for metrica in metricas:
        valores = evaluacion["test_" + metrica]

        fila[metrica + "_promedio"] = valores.mean()
        fila[metrica + "_desv"] = valores.std()

    resultados_cv.append(fila)

#Mostrar resultados
tabla_cv = pd.DataFrame(resultados_cv)

print("\nRESULTADOS DE VALIDACIÓN CRUZADA")
print(tabla_cv.round(4).to_string(index = False))


VALIDACIÓN CRUZADA (5-FOLD)

RESULTADOS DE VALIDACIÓN CRUZADA
             Modelo  accuracy_promedio  accuracy_desv  precision_promedio  precision_desv  recall_promedio  recall_desv  f1_promedio  f1_desv  roc_auc_promedio  roc_auc_desv
Regresión logística             0.8021         0.0118              0.6529          0.0262           0.5431       0.0411       0.5923   0.0299            0.8462        0.0126
      Random Forest             0.7852         0.0128              0.6260          0.0323           0.4742       0.0299       0.5394   0.0290            0.8178        0.0123


## Ajuste de hiperparámetros

In [9]:
#Definir hiperparámetros para cada modelo
parametros = {
    "Regresión logística": {
        "modelo__C": [0.1, 1, 10],
        "modelo__class_weight": [None, "balanced"]
    },
    "Random Forest": {
        "modelo__n_estimators": [100, 200],
        "modelo__max_depth": [5, 10, None],
        "modelo__class_weight": [None, "balanced"]
    }
}

#Buscar las mejores configuraciones
print("\nAJUSTE DE HIPERPARÁMETROS")

modelos_ajustados = {}

for nombre, modelo in modelos.items():

    busqueda = GridSearchCV(
        estimator = clone(modelo),
        param_grid = parametros[nombre],
        scoring = "f1",
        cv = kfold,
        n_jobs = 1
    )

    busqueda.fit(X_train, y_train)

    modelos_ajustados[nombre] = busqueda.best_estimator_

    print("\nModelo:", nombre)
    print("Mejores parámetros:", busqueda.best_params_)
    print("Mejor F1 promedio:", round(busqueda.best_score_, 4))


AJUSTE DE HIPERPARÁMETROS

Modelo: Regresión logística
Mejores parámetros: {'modelo__C': 0.1, 'modelo__class_weight': 'balanced'}
Mejor F1 promedio: 0.6295

Modelo: Random Forest
Mejores parámetros: {'modelo__class_weight': 'balanced', 'modelo__max_depth': 10, 'modelo__n_estimators': 100}
Mejor F1 promedio: 0.6282


## Evaluación de modelos ajustados

In [11]:
#Evaluar los modelos ajustados
print("\nEVALUACIÓN DE MODELOS AJUSTADOS")

resultados_ajustados = []

for nombre, modelo in modelos_ajustados.items():

    #Generar predicciones
    predicciones = modelo.predict(X_test)

    #Calcular probabilidades
    probabilidades = modelo.predict_proba(X_test)[:, 1]

    #Calcular métricas
    resultados_ajustados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, predicciones),
        "Precision": precision_score(y_test, predicciones),
        "Recall": recall_score(y_test, predicciones),
        "F1": f1_score(y_test, predicciones),
        "ROC-AUC": roc_auc_score(y_test, probabilidades)
    })

#Crear tabla comparativa
tabla_ajustada = pd.DataFrame(resultados_ajustados)

print("\nCOMPARACIÓN DE MODELOS AJUSTADOS")
print(tabla_ajustada.round(4).to_string(index = False))


EVALUACIÓN DE MODELOS AJUSTADOS

COMPARACIÓN DE MODELOS AJUSTADOS
             Modelo  Accuracy  Precision  Recall     F1  ROC-AUC
Regresión logística    0.7417     0.5087  0.7861 0.6176    0.841
      Random Forest    0.7686     0.5496  0.7112 0.6200    0.837


## Comparación y visualización

In [14]:
#Comparar resultados iniciales y ajustados
tabla_inicial = tabla.copy()
tabla_inicial["Versión"] = "Inicial"

tabla_final = tabla_ajustada.copy()
tabla_final["Versión"] = "Ajustado"

comparacion = pd.concat(
    [tabla_inicial, tabla_final],
    ignore_index = True
)

#Guardar resultados
carpeta = Path.cwd() / "resultados"
carpeta.mkdir(exist_ok = True)

comparacion.to_csv(
    carpeta / "comparacion_modelos.csv",
    index = False
)

#Crear gráfica comparativa
grafica = comparacion.set_index(
    ["Modelo", "Versión"]
)[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]]

grafica.plot(kind = "bar", figsize = (11, 6))

plt.title("Comparación de modelos iniciales y ajustados")
plt.ylabel("Valor de la métrica")
plt.xlabel("Modelo y versión")
plt.ylim(0, 1)
plt.xticks(rotation = 20, ha = "right")
plt.legend(title = "Métricas")
plt.tight_layout()

plt.savefig(
    carpeta / "comparacion_modelos.png",
    dpi=300
)

plt.close()

print("\nResultados comparativos guardados correctamente.")


Resultados comparativos guardados correctamente.
